In [2]:
from utils.spark_session import create_spark_session
spark = create_spark_session("verify-pos-counts")

gold = spark.read.format("delta").load("s3a://cs611-project/gold/pos_counts")
print(f"Total rows: {gold.count():,}")

# One row's POS-grouped lemma counts
row = gold.limit(1).collect()[0]
print(f"Document: {row['document_id']}")
print(f"Labels:   {row['labels']}")
print(f"POS tags present: {list(row['pos_counts'].keys())}")
for pos, counts in row['pos_counts'].items():
    top5 = sorted(counts.items(), key=lambda kv: -kv[1])[:5]
    print(f"  {pos}: top 5 = {top5}")

Total rows: 54,986


[Stage 50:>                                                         (0 + 1) / 1]

Document: 31984D0419
Labels:   Agriculture & Food Safety
POS tags present: ['AUX', 'PRON', 'CCONJ', 'PROPN', 'NUM', 'ADJ', 'SCONJ', 'ADP', 'DET', 'ADV', 'PART', 'VERB', 'NOUN']
  AUX: top 5 = [('be', 21), ('may', 5), ('have', 4), ('must', 3), ('shall', 2)]
  PRON: top 5 = [('it', 4), ('its', 1)]
  CCONJ: top 5 = [('and', 10), ('or', 4)]
  PROPN: top 5 = [('article', 8), ('commission', 5), ('member', 4), ('july', 4), ('eec', 3)]
  NUM: top 5 = [('two', 1), ('one', 1)]
  ADJ: top 5 = [('main', 5), ('special', 4), ('several', 2), ('pure', 2), ('supplementary', 2)]
  SCONJ: top 5 = [('whereas', 5), ('that', 2), ('where', 1), ('although', 1), ('so', 1)]
  ADP: top 5 = [('of', 31), ('in', 29), ('for', 10), ('to', 10), ('down', 8)]
  DET: top 5 = [('the', 55), ('a', 13), ('that', 5), ('this', 3), ('whose', 3)]
  ADV: top 5 = [('only', 1), ('therefore', 1), ('whereas', 1), ('thereof', 1), ('last', 1)]
  PART: top 5 = [('to', 4), ('not', 2)]
  VERB: top 5 = [('enter', 9), ('lay', 8), ('meet', 4

In [6]:
gold.select("document_id", "labels", "snapshot_date", "n_unique_tokens", "n_total_tokens") \
    .show(20, truncate=80)

[Stage 53:>                                                         (0 + 1) / 1]

+-----------+----------------------------------------------------------------------------+-------------+---------------+--------------+
|document_id|                                                                      labels|snapshot_date|n_unique_tokens|n_total_tokens|
+-----------+----------------------------------------------------------------------------+-------------+---------------+--------------+
| 31984D0419|                                                   Agriculture & Food Safety|   1984-01-01|            199|           595|
| 31984D0247|Labour, Employment & Social; Environment & Energy; Agriculture & Food Safety|   1984-01-01|            207|           704|
| 31984D0309|          Agriculture & Food Safety; Trade & Customs; International & EU Law|   1984-01-01|            132|           293|
| 31984D0371|       Agriculture & Food Safety; Industry, Technology & IP; Trade & Customs|   1984-01-01|            139|           327|
| 31984D0561|       Agriculture & Food Safety; I

In [7]:
row = gold.limit(1).collect()[0]

print(f"document_id: {row['document_id']}")
print(f"labels:      {row['labels']}")
print(f"snapshot_date: {row['snapshot_date']}")
print(f"n_unique_tokens: {row['n_unique_tokens']}")
print(f"n_total_tokens:  {row['n_total_tokens']}")

print("\npos_counts (top 10 per POS tag):")
for pos, counts in row['pos_counts'].items():
    top = sorted(counts.items(), key=lambda kv: -kv[1])[:10]
    print(f"\n  {pos}:")
    for lemma, cnt in top:
        print(f"    {cnt:4d}  {lemma}")

[Stage 56:>                                                         (0 + 1) / 1]

document_id: 31984D0419
labels:      Agriculture & Food Safety
snapshot_date: 1984-01-01
n_unique_tokens: 199
n_total_tokens:  595

pos_counts (top 10 per POS tag):

  AUX:
      21  be
       5  may
       4  have
       3  must
       2  shall
       1  will
       1  should
       1  do

  PRON:
       4  it
       1  its

  CCONJ:
      10  and
       4  or

  PROPN:
       8  article
       5  commission
       4  member
       4  july
       3  eec
       2  states
       2  greece
       2  european
       2  l
       2  directive

  NUM:
       1  two
       1  one

  ADJ:
       5  main
       4  special
       2  several
       2  pure
       2  supplementary
       2  fourth
       1  second
       1  same
       1  commission
       1  portuguese

  SCONJ:
       5  whereas
       2  that
       1  where
       1  although
       1  so
       1  whether

  ADP:
      31  of
      29  in
      10  for
      10  to
       8  down
       6  with
       4  as
       4  by
     

In [8]:
import pandas as pd
sample = gold.limit(10).toPandas()
sample

,document_id,labels,snapshot_date,pos_counts,n_unique_tokens,n_total_tokens
0,31984D0419,Agriculture & Food Safety,1984-01-01,"{'AUX': {'be': 21, 'may': 5, 'will': 1, 'have'...",199,595
1,31984D0247,"Labour, Employment & Social; Environment & Ene...",1984-01-01,"{'AUX': {'have': 8, 'can': 2, 'shall': 2, 'mus...",207,704
2,31984D0309,Agriculture & Food Safety; Trade & Customs; In...,1984-01-01,"{'DET': {'these': 2, 'which': 1, 'the': 30, 't...",132,293
3,31984D0371,"Agriculture & Food Safety; Industry, Technolog...",1984-01-01,"{'PRON': {'i': 1, 'it': 1}, 'AUX': {'have': 1,...",139,327
4,31984D0561,"Agriculture & Food Safety; Industry, Technolog...",1984-01-01,"{'DET': {'the': 34, 'no': 1, 'that': 1, 'a': 1...",98,234
5,31984D0560,"Agriculture & Food Safety; Industry, Technolog...",1984-01-01,"{'DET': {'the': 34, 'no': 1, 'that': 1, 'a': 1...",95,224
6,31984D0375,"Finance, Tax & Banking; Agriculture & Food Saf...",1984-01-01,"{'AUX': {'have': 1, 'should': 2, 'shall': 2, '...",140,399
7,31985D0071,"Environment & Energy; Industry, Technology & I...",1984-01-01,"{'PRON': {'i': 4, 'its': 7, 'itself': 1, 'it':...",216,753
8,31985D0046,"Labour, Employment & Social; Agriculture & Foo...",1984-01-01,"{'PRON': {'i': 1}, 'AUX': {'should': 1, 'have'...",393,1826
9,31984D0589,Agriculture & Food Safety; Economic & Competit...,1984-01-01,"{'DET': {'which': 1, 'the': 34, 'this': 6, 'a'...",122,313
